## DSL Computation: Bias-Corrected Prevalence Estimates for CARDS Superclaims

This notebook implements the Design-Based Supervised Learning (DSL) estimator proposed by Egami et al. (2024) to estimate the true prevalence of each superclaim category in the corpus, correcting for systematic prediction error introduced by automated CARDS annotation.

**Reference:**
> Egami, N., Hinck, M., Stewart, B. M., & Wei, H. (2024). Using Large Language Model Annotations for the Social Sciences: A General Framework of Using Predicted Variables in Downstream Analyses.

### Inputs and Outputs

**Inputs:**
- `llm_annotations.csv` — CARDS annotations for the full corpus
- `manual_annotations.xlsx` — Expert annotations for the 500-article validation sample

**Outputs:**
- Results table printed inline
- `dsl_results.csv`

## Imports and Configuration

In [38]:
import ast
import numpy as np
import pandas as pd

# file paths
LLM_PATH = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/annotated/llm_annotations.csv"
MANUAL_PATH = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/annotated/manual_annotations.xlsx"
OUTPUT_PATH = "dsl_results.csv"

# superclaim category definitions
SUPERCLAIM_NAMES = {
    0: "No claim detected",
    1: "Global warming not happening",
    2: "Humans not causing climate change",
    3: "Climate impacts not bad",
    4: "Climate solutions harmful/unnecessary",
    5: "Climate science unreliable",
    6: "Climate proponents alarmist/biased",
    7: "We need fossil fuels",
}

SUPERCLAIMS = list(SUPERCLAIM_NAMES.keys())

## Data Loading and Parsing

In [39]:
def parse_llm_claims(val):
    """
    Parse CARDS superclaim labels from the LLM annotations file.
    The CardsClaim column stores superclaim numbers as a list, e.g. [1, 4].
    Returns an empty list if the value is missing or unparseable.
    """
    if pd.isna(val):
        return []
    try:
        return ast.literal_eval(str(val))
    except (ValueError, SyntaxError):
        return []


def parse_manual_claims(val):
    """
    Parse superclaim labels from the manual annotations file.
    The ClaimCat column stores values like '0_0_0' or '4_0_0; 7_0_0'.
    Extracts the leading integer from each semicolon-separated entry.
    Returns an empty list if the value is missing or unparseable.
    """
    if pd.isna(val):
        return []
    parts = [p.strip() for p in str(val).split(";")]
    claims = []
    for part in parts:
        prefix = part.split("_")[0]
        try:
            claims.append(int(prefix))
        except ValueError:
            pass
    return list(set(claims))

In [40]:
# load raw data
llm = pd.read_csv(LLM_PATH)
manual = pd.read_excel(MANUAL_PATH)

print(f"LLM annotations loaded:    {len(llm):,} articles")
print(f"Manual annotations loaded: {len(manual):,} articles")

LLM annotations loaded:    17,975 articles
Manual annotations loaded: 500 articles


In [41]:
# parse claim labels and create binary indicators for each superclaim
llm["claims_parsed"] = llm["CardsClaim"].apply(parse_llm_claims)
manual["claims_parsed"] = manual["ClaimCat"].apply(parse_manual_claims)

for s in SUPERCLAIMS:
    llm[f"llm_{s}"] = llm["claims_parsed"].apply(lambda x: 1 if s in x else 0)
    manual[f"manual_{s}"] = manual["claims_parsed"].apply(lambda x: 1 if s in x else 0)
 
# sanity check: verify parsing for a few articles
print("LLM PARSING CHECK")
print("Raw CardsClaim vs parsed claims for first 5 articles:")
print(llm[["ArticleKey", "CardsClaim", "claims_parsed"]].head(5).to_string())

print()
print("MANUAL PARSING CHECK")
print("Raw ClaimCat vs parsed claims for first 5 articles:")
print(manual[["ArticleKey", "ClaimCat", "claims_parsed"]].head(5).to_string())


LLM PARSING CHECK
Raw CardsClaim vs parsed claims for first 5 articles:
  ArticleKey CardsClaim claims_parsed
0   e9e57e17        [1]           [1]
1   e9620bc1     [3, 4]        [3, 4]
2   e996a478        [4]           [4]
3   e9ca9690        [4]           [4]
4   e9cc2f19     [3, 6]        [3, 6]

MANUAL PARSING CHECK
Raw ClaimCat vs parsed claims for first 5 articles:
  ArticleKey ClaimCat claims_parsed
0   eb0e1b6e    0_0_0           [0]
1   eac48880    0_0_0           [0]
2   e9dd5e38    0_0_0           [0]
3   ea10e471    0_0_0           [0]
4   ea3ad383    0_0_0           [0]


In [42]:
# merge to align CARDS and expert labels for the validation sample
llm_cols = ["ArticleKey"] + [f"llm_{s}" for s in SUPERCLAIMS]
manual_cols = ["ArticleKey"] + [f"manual_{s}" for s in SUPERCLAIMS]
merged = manual[manual_cols].merge(llm[llm_cols], on="ArticleKey", how="left")

merged.head(5)

,ArticleKey,manual_0,manual_1,manual_2,manual_3,manual_4,manual_5,manual_6,manual_7,llm_0,llm_1,llm_2,llm_3,llm_4,llm_5,llm_6,llm_7
0,eb0e1b6e,1,0,0,0,0,0,0,0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,eac48880,1,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,e9dd5e38,1,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,ea10e471,1,0,0,0,0,0,0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,ea3ad383,1,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [44]:
# convert llm columns to integers 
for s in SUPERCLAIMS:
    merged[f"llm_{s}"] = merged[f"llm_{s}"].fillna(0).astype(int)
    
merged.head(5)

,ArticleKey,manual_0,manual_1,manual_2,manual_3,manual_4,manual_5,manual_6,manual_7,llm_0,llm_1,llm_2,llm_3,llm_4,llm_5,llm_6,llm_7
0,eb0e1b6e,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
1,eac48880,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
2,e9dd5e38,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
3,ea10e471,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
4,ea3ad383,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0


In [45]:
print("BINARY INDICATOR CHECK")
print("Verify binary indicators match parsed claims for first 3 LLM articles:")
for i, row in llm.head(3).iterrows():
    active = [s for s in SUPERCLAIMS if row[f"llm_{s}"] == 1]
    print(f"  ArticleKey: {row['ArticleKey']} | Raw: {row['CardsClaim']} | Parsed: {row['claims_parsed']} | Active superclaims: {active}")
print()
print("Verify binary indicators match parsed claims for first 3 manual articles:")
for i, row in manual.head(3).iterrows():
    active = [s for s in SUPERCLAIMS if row[f"manual_{s}"] == 1]
    print(f"  ArticleKey: {row['ArticleKey']} | Raw: {row['ClaimCat']} | Parsed: {row['claims_parsed']} | Active superclaims: {active}")

BINARY INDICATOR CHECK
Verify binary indicators match parsed claims for first 3 LLM articles:
  ArticleKey: e9e57e17 | Raw: [1] | Parsed: [1] | Active superclaims: [1]
  ArticleKey: e9620bc1 | Raw: [3, 4] | Parsed: [3, 4] | Active superclaims: [3, 4]
  ArticleKey: e996a478 | Raw: [4] | Parsed: [4] | Active superclaims: [4]

Verify binary indicators match parsed claims for first 3 manual articles:
  ArticleKey: eb0e1b6e | Raw: 0_0_0 | Parsed: [0] | Active superclaims: [0]
  ArticleKey: eac48880 | Raw: 0_0_0 | Parsed: [0] | Active superclaims: [0]
  ArticleKey: e9dd5e38 | Raw: 0_0_0 | Parsed: [0] | Active superclaims: [0]


## Label Distributions

Before running the DSL estimation, it is useful to inspect the raw label distributions from both CARDS and manual coding.

In [46]:
print("CARDS label distribution (full corpus):")
for s in SUPERCLAIMS:
    count = llm[f"llm_{s}"].sum()
    pct = llm[f"llm_{s}"].mean() * 100
    print(f"  {s}: {SUPERCLAIM_NAMES[s]:<45} {count:>6,} ({pct:.1f}%)")

print()
print("Manual label distribution (500-article validation sample):")
for s in SUPERCLAIMS:
    count = manual[f"manual_{s}"].sum()
    pct = manual[f"manual_{s}"].mean() * 100
    print(f"  {s}: {SUPERCLAIM_NAMES[s]:<45} {count:>6,} ({pct:.1f}%)")

CARDS label distribution (full corpus):
  0: No claim detected                                911 (5.1%)
  1: Global warming not happening                   2,827 (15.7%)
  2: Humans not causing climate change                473 (2.6%)
  3: Climate impacts not bad                        2,871 (16.0%)
  4: Climate solutions harmful/unnecessary         14,750 (82.1%)
  5: Climate science unreliable                       337 (1.9%)
  6: Climate proponents alarmist/biased               536 (3.0%)
  7: We need fossil fuels                             789 (4.4%)

Manual label distribution (500-article validation sample):
  0: No claim detected                                451 (90.2%)
  1: Global warming not happening                       1 (0.2%)
  2: Humans not causing climate change                  1 (0.2%)
  3: Climate impacts not bad                            1 (0.2%)
  4: Climate solutions harmful/unnecessary             41 (8.2%)
  5: Climate science unreliable                    

## DSL Estimation

In [ ]:
# define sampling parameters
N = len(llm)    # total corpus size
n = len(merged) # validation sample size
pi = n / N      # sampling probability under simple random sampling

print(f"Total corpus size (N):         {N:,}")
print(f"Validation sample size (n):    {n:,}")
print(f"Sampling probability (π):      {pi:.6f}")

Total corpus size (N):         17,975
Validation sample size (n):    500
Sampling probability (π):      0.027816


In [48]:
results = []

for s in SUPERCLAIMS:

    # Step 1: CARDS mean over full corpus
    cards_mean_full = llm[f"llm_{s}"].mean()

    # Step 2: CARDS mean over validation sample
    cards_mean_sample = merged[f"llm_{s}"].mean()

    # Step 3: Manual mean over validation sample
    manual_mean_sample = merged[f"manual_{s}"].mean()

    # Step 4: Estimated bias
    bias = cards_mean_sample - manual_mean_sample

    # Step 5: DSL-corrected estimate
    dsl_estimate = cards_mean_full - bias

    # Step 6: Construct design-adjusted outcomes (Y_tilde) for all N articles
    # Start with CARDS predictions for all articles
    y_tilde = llm[f"llm_{s}"].copy().values.astype(float)
    

# Apply bias correction for articles in the validation sample
    for _, row in merged.iterrows():
        match = llm[llm["ArticleKey"] == row["ArticleKey"]]
        if len(match) > 0:
            corpus_idx = match.index[0]
            y_hat = row[f"llm_{s}"]
            y = row[f"manual_{s}"]
            y_tilde[corpus_idx] = y_hat - (1 / pi) * (y_hat - y)

    # Step 7: Standard error and 95% confidence interval
    se = np.std(y_tilde, ddof=1) / np.sqrt(N)
    ci_lower = max(dsl_estimate - 1.96 * se, 0)  # Floor at zero
    ci_upper = dsl_estimate + 1.96 * se

    results.append({
        "superclaim": s,
        "name": SUPERCLAIM_NAMES[s],
        "cards_estimate": cards_mean_full,
        "manual_estimate": manual_mean_sample,
        "bias": bias,
        "dsl_estimate": max(dsl_estimate, 0),  # Floor at zero
        "se": se,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
    })

print("DSL estimation complete.")

DSL estimation complete.


## Results

In [49]:
# Display results as a formatted dataframe
results_df = pd.DataFrame(results)

display_df = pd.DataFrame({
    "Superclaim": [f"{r['superclaim']}: {r['name']}" for r in results],
    "CARDS (%)": [f"{r['cards_estimate']*100:.1f}" for r in results],
    "Manual (%)": [f"{r['manual_estimate']*100:.1f}" for r in results],
    "Bias (pp)": [f"{r['bias']*100:+.1f}" for r in results],
    "DSL estimate (%)": [f"{r['dsl_estimate']*100:.1f}" for r in results],
    "95% CI": [f"[{r['ci_lower']*100:.1f}, {r['ci_upper']*100:.1f}]" for r in results],
})

display_df.set_index("Superclaim", inplace=True)
display_df

,CARDS (%),Manual (%),Bias (pp),DSL estimate (%),95% CI
Superclaim,,,,,
0: No claim detected,5.1,90.2,-85.0,90.1,"[82.1, 98.0]"
1: Global warming not happening,15.7,0.2,+17.2,0.0,"[0.0, 2.1]"
2: Humans not causing climate change,2.6,0.2,+2.6,0.0,"[0.0, 1.5]"
3: Climate impacts not bad,16.0,0.2,+15.2,0.8,"[0.0, 4.2]"
4: Climate solutions harmful/unnecessary,82.1,8.2,+72.8,9.3,"[1.9, 16.6]"
5: Climate science unreliable,1.9,0.8,+1.0,0.9,"[0.0, 2.3]"
6: Climate proponents alarmist/biased,3.0,1.2,+0.8,2.2,"[0.7, 3.6]"
7: We need fossil fuels,4.4,1.0,+2.8,1.6,"[0.0, 3.3]"


In [50]:
# Save results to CSV
output_df = results_df.copy()
for col in ["cards_estimate", "manual_estimate", "bias", "dsl_estimate", "se", "ci_lower", "ci_upper"]:
    output_df[col] = output_df[col] * 100

output_df.columns = [
    "superclaim", "name",
    "cards_pct", "manual_pct", "bias_pp",
    "dsl_pct", "se_pct", "ci_lower_pct", "ci_upper_pct"
]

output_df.to_csv(OUTPUT_PATH, index=False)
print(f"Results saved to {OUTPUT_PATH}")

Results saved to dsl_results.csv
